# GeoNote — ข้อมูลออฟไลน์ทั้งประเทศ แบ่งตามภาค
สร้างไฟล์ทั้งหมดที่แอปใช้ในรอบเดียว:

| ไฟล์ | คืออะไร |
|---|---|
| `data/geol/<ภาค>.geojson` | หน่วยหิน 1:250,000 ของกรมทรัพยากรธรณี (ROCK_UNIT_250K) |
| `data/rock_colors.json` | สีตาม legend ของกรมทรัพยากรธรณี |
| `data/map/<ภาค>.pmtiles.001 …` | แผนที่ถนนออฟไลน์จากข้อมูล OpenStreetMap (ผ่าน Protomaps) ภาคละไม่เกิน 95 MB แบ่งเป็นชิ้นละ 20 MB |
| `data/regions.json` | รายการไฟล์และขนาด ให้แอปรู้ว่ามีภาคไหนบนเว็บ |

ภาค: `isan` `north` `west` `central` `east` `south` (ต้องตรงกับ `js/regions.js` ในแอป)

**วิธีใช้:** Runtime → Run all (แผนที่ใช้เวลาราว 20–60 นาที) ผลลัพธ์อยู่ใน Google Drive `MyDrive/GeoNote_data/data/` คัดลอกโฟลเดอร์ `data` ไปทับใน `geo/` ของแอป แล้วอัปขึ้น GitHub ได้ทั้งผ่านหน้าเว็บ (ทุกไฟล์เล็กกว่า 25 MB) หรือ GitHub Desktop

ข้อมูล © กรมทรัพยากรธรณี (CC BY) · © OpenStreetMap contributors (ODbL)

In [ ]:
# ---------- settings ----------
BUILD_ROCKS = True        # หน่วยหินทั้งประเทศ
BUILD_MAPS = True         # แผนที่ออฟไลน์
MAP_REGIONS = ["isan", "north", "west", "central", "east", "south"]   # ลดรายการได้ถ้าจะทำบางภาค
MAX_MB = 95               # GitHub รับไฟล์ละไม่เกิน 100 MB
SIMPLIFY_DEG = 0.0002     # ลดจุดของขอบหน่วยหิน (~20 m)
SAVE_TO_DRIVE = True

# same order and boxes as js/regions.js — the first box that contains a place wins
REGIONS = [
    ("isan",    (100.8, 14.0, 105.7, 18.5)),
    ("north",   (97.3, 16.5, 101.5, 20.5)),
    ("west",    (97.3, 11.5, 99.6, 16.5)),
    ("central", (99.6, 12.0, 100.85, 16.5)),
    ("east",    (100.85, 11.5, 103.0, 14.0)),
    ("south",   (97.3, 5.6, 102.2, 11.5)),
]
THAILAND = (97.3, 5.6, 105.7, 20.5)

import os, json, requests, datetime
import geopandas as gpd, pandas as pd
from shapely.geometry import box, mapping
from shapely.ops import unary_union

ROOT = "/content/GeoNote_data"
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = "/content/drive/MyDrive/GeoNote_data"
    except Exception as e:
        SAVE_TO_DRIVE = False
        print("⚠ เชื่อม Google Drive ไม่ได้:", e)
        print("  ไม่เป็นไร รันต่อได้ ไฟล์จะเก็บไว้ใน Colab แล้วดาวน์โหลดให้ทีละไฟล์ในเซลล์สุดท้าย")
DATA = f"{ROOT}/data"
os.makedirs(f"{DATA}/geol", exist_ok=True)
os.makedirs(f"{DATA}/map", exist_ok=True)

# each region's own area = its box minus the boxes before it
AREAS, taken = {}, None
for rid, b in REGIONS:
    g = box(*b)
    AREAS[rid] = g if taken is None else g.difference(taken)
    taken = g if taken is None else unary_union([taken, g])
print("output:", DATA)

## ส่วนที่ 1 · หน่วยหิน 1:250,000 ทั้งประเทศ

In [ ]:
SERVICE = "https://gisportal.dmr.go.th/arcgis/rest/services/GEOL/ROCK_UNIT_250K/MapServer/0"

if BUILD_ROCKS:
    q = dict(geometry=",".join(map(str, THAILAND)), geometryType="esriGeometryEnvelope", inSR=4326,
             spatialRel="esriSpatialRelIntersects", returnIdsOnly="true", f="json")
    ids = sorted(requests.get(f"{SERVICE}/query", params=q, timeout=180).json()["objectIds"])
    print("units in Thailand:", len(ids))

    parts = []
    for i in range(0, len(ids), 200):
        r = requests.post(f"{SERVICE}/query", timeout=300, data=dict(
            objectIds=",".join(map(str, ids[i:i + 200])), outFields="*", outSR=4326, f="geojson"))
        parts.append(gpd.GeoDataFrame.from_features(r.json()["features"], crs=4326))
        print(f"  {min(i + 200, len(ids))}/{len(ids)}", end="\r")
    rocks = pd.concat(parts, ignore_index=True)
    rocks = rocks[rocks.geometry.notna() & ~rocks.geometry.is_empty]
    if SIMPLIFY_DEG:
        rocks["geometry"] = rocks.geometry.simplify(SIMPLIFY_DEG, preserve_topology=True)
    keep = ["OBJECTID", "TEXT_SYM"] + [c for c in rocks.columns
            if c not in ("OBJECTID", "TEXT_SYM", "geometry") and rocks[c].dtype == object
            and not c.upper().startswith(("SHAPE", "GLOBALID"))]
    rocks = rocks[keep + ["geometry"]]
    print("\nfields:", keep)

In [ ]:
# each unit goes to ONE region: the one holding a point inside the unit (no clipping, no duplicates)
if BUILD_ROCKS:
    def region_of(geom):
        p = geom.representative_point()
        for rid, area in AREAS.items():
            if area.contains(p):
                return rid
        return min(AREAS, key=lambda rid: AREAS[rid].distance(p))   # islands / edges
    rocks["region"] = rocks.geometry.apply(region_of)

    geol_manifest = {}
    for rid, _ in REGIONS:
        sub = rocks[rocks.region == rid].drop(columns="region")
        path = f"{DATA}/geol/{rid}.geojson"
        sub.to_file(path, driver="GeoJSON", COORDINATE_PRECISION=5)
        geol_manifest[rid] = {"file": f"data/geol/{rid}.geojson", "bytes": os.path.getsize(path), "units": len(sub)}
        print(f"{rid:8s} {len(sub):5d} units  {os.path.getsize(path)/1e6:6.1f} MB")

    info = requests.get(SERVICE, params=dict(f="json"), timeout=60).json()
    colors = {str(v["value"]).strip(): "#%02X%02X%02X" % tuple(v["symbol"]["color"][:3])
              for v in info["drawingInfo"]["renderer"].get("uniqueValueInfos", []) if v.get("symbol", {}).get("color")}
    json.dump(colors, open(f"{DATA}/rock_colors.json", "w"), ensure_ascii=False)
    print("colours:", len(colors))

## ส่วนที่ 2 · แผนที่ OpenStreetMap ออฟไลน์ตามภาค
ตัดจากแผนที่โลกรายวันของ Protomaps (ข้อมูล OSM) ไม่ได้ดึงจาก tile.openstreetmap.org ซึ่งห้ามดาวน์โหลดล่วงหน้า

แต่ละภาคลองที่ zoom 14 ก่อน ถ้าไฟล์เกิน MAX_MB จะลดเป็น 13 และ 12 เอง (zoom 14 ≈ รายละเอียดเกือบเท่าแผนที่ออนไลน์, 13 ≈ ถนนในหมู่บ้าน, 12 ≈ ถนนหลักและเมือง)

In [ ]:
import tarfile, io, subprocess
if BUILD_MAPS:
    rel = requests.get("https://api.github.com/repos/protomaps/go-pmtiles/releases/latest", timeout=60).json()
    asset = next(a for a in rel["assets"] if a["name"].endswith("_Linux_x86_64.tar.gz"))
    tarfile.open(fileobj=io.BytesIO(requests.get(asset["browser_download_url"], timeout=300).content)).extract("pmtiles", ".")
    os.chmod("pmtiles", 0o755)
    print(subprocess.run(["./pmtiles", "version"], capture_output=True, text=True).stdout)

    SRC = None
    try:
        builds = requests.get("https://build-metadata.protomaps.dev/builds.json", timeout=60).json()
        SRC = "https://build.protomaps.com/" + sorted(b["key"] for b in builds)[-1]
    except Exception as e:
        print("builds.json:", e)
        for d in range(15):
            day = (datetime.date.today() - datetime.timedelta(days=d)).strftime("%Y%m%d")
            if requests.head(f"https://build.protomaps.com/{day}.pmtiles", timeout=30).ok:
                SRC = f"https://build.protomaps.com/{day}.pmtiles"; break
    print("source:", SRC)

In [ ]:
map_manifest = {}
if BUILD_MAPS:
    for rid in MAP_REGIONS:
        area = f"/content/{rid}_area.geojson"
        json.dump({"type": "Feature", "properties": {}, "geometry": mapping(AREAS[rid])}, open(area, "w"))
        out = f"/content/{rid}.pmtiles"
        for z in (14, 13, 12, 11):
            if os.path.exists(out): os.remove(out)
            r = subprocess.run(["./pmtiles", "extract", SRC, out, f"--region={area}", f"--maxzoom={z}"],
                               capture_output=True, text=True)
            if r.returncode != 0:
                print(r.stderr[-2000:]); raise SystemExit(f"extract failed for {rid}")
            mb = os.path.getsize(out) / 1e6
            print(f"{rid:8s} z{z}: {mb:6.1f} MB")
            if mb <= MAX_MB: break
        # GitHub's web upload takes files under 25 MB, so each map is saved as 20 MB pieces
        # (<id>.pmtiles.001, .002 …); the app downloads the pieces and joins them
        import glob, math
        for old in glob.glob(f"{DATA}/map/{rid}.pmtiles*"):
            os.remove(old)
        data = open(out, "rb").read()
        PART = 20 * 1000 * 1000
        parts = []
        for i in range(math.ceil(len(data) / PART)):
            name = f"{rid}.pmtiles.{i + 1:03d}"
            open(f"{DATA}/map/{name}", "wb").write(data[i * PART:(i + 1) * PART])
            parts.append(f"data/map/{name}")
        map_manifest[rid] = {"parts": parts, "bytes": len(data), "maxzoom": z}
        print(f"         → {len(parts)} pieces")

## ส่วนที่ 3 · รายการไฟล์ (regions.json)

In [ ]:
mpath = f"{DATA}/regions.json"
old = json.load(open(mpath)) if os.path.exists(mpath) else {}
manifest = {
    "generated": datetime.datetime.now().isoformat(timespec="minutes"),
    "note": "Written by GeoNote_offline_data.ipynb. Lists the files on the website.",
    "geol": geol_manifest if BUILD_ROCKS else old.get("geol", {}),
    "map": {**old.get("map", {}), **map_manifest},
}
json.dump(manifest, open(mpath, "w"), ensure_ascii=False, indent=1)
print(json.dumps(manifest, ensure_ascii=False, indent=1))
total = sum(v["bytes"] for k in ("geol", "map") for v in manifest[k].values()) / 1e6
print(f"\nรวม {total:.0f} MB → {DATA}")
print("คัดลอกโฟลเดอร์ data ทั้งหมดไปทับใน geo/data ของแอป แล้ว push ขึ้น GitHub")

## ส่วนที่ 4 · ดาวน์โหลด (เฉพาะตอนเชื่อม Google Drive ไม่ได้)
ถ้าเก็บลง Drive ได้ เซลล์นี้จะไม่ทำอะไร ถ้าไม่ได้ จะดาวน์โหลดให้ 2 แบบ:
1. `geonote_data_small.zip` = regions.json + rock_colors.json + หน่วยหินทุกภาค แตกไฟล์แล้ววางทับ `geo/` (ได้โฟลเดอร์ `data/...`)
2. ชิ้นแผนที่ `<ภาค>.pmtiles.001 …` ทีละไฟล์ วางใน `geo/data/map/`

เบราว์เซอร์อาจถามว่า "อนุญาตให้ดาวน์โหลดหลายไฟล์" ให้กดอนุญาต ไฟล์ละไม่เกิน 20 MB

In [ ]:
if not SAVE_TO_DRIVE:
    import zipfile, time
    from google.colab import files
    small = "/content/geonote_data_small.zip"
    with zipfile.ZipFile(small, "w", zipfile.ZIP_DEFLATED) as z:
        for rel in ["regions.json", "rock_colors.json"] + [f"geol/{rid}.geojson" for rid, _ in REGIONS]:
            if os.path.exists(f"{DATA}/{rel}"):
                z.write(f"{DATA}/{rel}", f"data/{rel}")
    print("zip:", round(os.path.getsize(small) / 1e6, 1), "MB"); files.download(small)
    import glob
    for p in sorted(glob.glob(f"{DATA}/map/*.pmtiles.*")):
        time.sleep(2); print("download", os.path.basename(p)); files.download(p)
else:
    print("ไฟล์อยู่ใน Google Drive แล้ว:", DATA)